In [0]:
%sql
CREATE OR REPLACE TABLE spotify.silver.artists AS

WITH artistas_recentes AS (
  SELECT
    id,
    name,
    type,
    try_element_at(images, 1) AS images,
    _ingestion_timestamp
  FROM spotify.bronze.artists
  WHERE id IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY _ingestion_timestamp DESC) = 1
)
SELECT
    id AS artist_id,
    name,
    type,
    images.height AS image_height,
    images.url AS image_url,
    images.width AS image_width,
    _ingestion_timestamp,
    current_timestamp() AS _processed_at
FROM artistas_recentes
ORDER BY ID;

SELECT * FROM spotify.silver.artists ORDER BY name, image_width DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE spotify.silver.albums AS

WITH albums_recentes AS (
  SELECT
    id AS album_id,
    artist_id,
    name AS album_name,
    release_date,
    total_tracks,
    try_element_at(images, 1) AS image,
    _ingestion_timestamp
  FROM spotify.bronze.albums
  WHERE id IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY _ingestion_timestamp DESC) = 1
)
SELECT
  album_id,
  artist_id,
  album_name,
  release_date,
  total_tracks,
  image.height AS image_height,
  image.url AS image_url,
  image.width AS image_width,
  _ingestion_timestamp,
  current_timestamp() AS _processed_at
FROM albums_recentes
ORDER BY ALBUM_ID;

SELECT * FROM spotify.silver.albums

In [0]:
%sql
CREATE OR REPLACE TABLE spotify.silver.album_tracks AS

WITH albums_recentes AS (
  SELECT
    id AS track_id,
    album_id,
    disc_number,
    duration_ms as music_duration_ms,
    explicit,
    name AS music_name,
    external_urls.spotify AS music_url,
    track_number as music_number,
    _ingestion_timestamp
  FROM spotify.bronze.album_tracks
  WHERE id IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY _ingestion_timestamp DESC) = 1
)
SELECT
    track_id,
    album_id,
    disc_number,
    music_duration_ms,
    explicit,
    music_name,
    music_url,
    music_number,
    _ingestion_timestamp
FROM albums_recentes
ORDER BY album_id;

SELECT * FROM spotify.silver.album_tracks